<a href="https://colab.research.google.com/github/mahidodiya/AI-ML/blob/main/Moderation_Simple_v5_1_Star_of_books_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content Moderation: fetch real posts, check SAFE / UNSAFE

Simple version. It does three things:
1. fetches a user's posts from the backend,
2. checks each post (caption + every image / video / audio file) with the moderation models,
3. shows one decision per post: `SAFE`, `REVIEW` (needs a human look, or something could not be checked) or `UNSAFE`.

Hosting (webhook server) comes after this works.

**Run order:** 1 Config, 2 Install, 3 Login, 4 Load models, 5 to 8 moderation code, 9 backend client, 10 Fetch, 11 Moderate.

**Before you start**, in the Colab sidebar open the key icon (Secrets) and add `EVOLVE_TOKEN` (the backend login token) and `HF_TOKEN` (Hugging Face), notebook access ON. The backend answers **401** without a valid token.


## 1. Config and secrets
The only place you need to edit (`USER_ID`, `POST_IDS`).

In [1]:
import os, json
from types import SimpleNamespace

def get_secret(name, prompt=None, required=True):
    """environment variable -> Colab Secrets -> hidden prompt. Secrets are never written in the notebook."""
    v = os.environ.get(name)
    if not v:
        try:
            from google.colab import userdata
            v = userdata.get(name)
        except Exception:
            v = None
    if not v and prompt:
        from getpass import getpass
        v = getpass(prompt)
    v = (v or "").strip().strip("\"'").strip()
    if v.lower().startswith("bearer "):
        v = v[7:].strip()
    if required and not v:
        raise RuntimeError(f"Missing secret {name}: add it in Colab Secrets (key icon) and turn notebook access ON.")
    return v or None

CFG = SimpleNamespace(
    # ---- models ----
    SAFETY_ID="nvidia/Nemotron-3.5-Content-Safety",
    SAFETY_DTYPE="auto",             # "auto" = test float16 on GPUs without bf16 (T4) and keep it only if answers match. Or "float16" / "bfloat16"
    TRY_FP16_EAGER=True,             # if float16+SDPA fails validation, try float16+eager attention once
    MODEL_CACHE_FILE="/content/model_config.json",   # remembers the validated precision, so later starts skip the ~1 min test (point it at Drive to survive resets)
    ASR_MODEL="large-v3-turbo",
    ASR_LANGUAGE=None,               # None = detect (restricted to Indian languages + English)
    ASR_BEAM=3,
    TRY_FASTER_WHISPER=True,

    # ---- speed knobs ----
    LATENCY_BUDGET_S=15,             # target seconds per video; the planner turns it into a number of Nemotron calls
    MONTAGE_TILES=4,                 # frames packed per Nemotron call (flagged montages are re-checked frame by frame)
    MAX_FRAMES=32,
    MIN_FRAME_GAP_S=1.0,
    MAX_FRAME_GAP_S=10.0,            # (triage OFF only)
    FAST_SEEK=True,

    # ---- frame triage (tiny CLIP ranks frames; Nemotron sees the riskiest of each time segment) ----
    TRIAGE=True,
    TRIAGE_MODEL="openai/clip-vit-base-patch32",
    TRIAGE_FRAMES=32,
    TRIAGE_ALWAYS_SEND=0.5,
    TRIAGE_ALWAYS_MAX=8,

    # ---- limits ----
    MAX_SECONDS=600,                 # longer video/audio: only the first MAX_SECONDS are checked (result is REVIEW, not SAFE)
    MAX_CHUNKS=100,
    MAX_TEXT_CHARS=2000,
    FRAME_SHORT_SIDE=512,
    FRAME_DEDUPE_DIFF=4.0,
    MIN_FLAGGED_FRAMES=1,            # 2 = a single odd frame becomes REVIEW instead of UNSAFE
    CLASSIFY_BATCH=4,
    SAFETY_MAX_NEW=32,

    # ---- decision policy ----
    SOFT_CATEGORIES={"needs caution"},
    LEXICON_ACTION="UNSAFE",         # "UNSAFE" or "REVIEW"
    LEXICON_FILE=None,               # optional text file, one extra term per line

    # ---- platform / backend ----
    API_BASE="https://evolve-backend-4lux.onrender.com",
    USER_ID=3,                       # whose posts to fetch
    POST_IDS=None,                   # e.g. [2, 5] to check only these posts (None = all)
    POSTS_ENDPOINT="/api/skills/user/{user_id}/skills-posts",
    REQUEST_TIMEOUT=90,              # Render free instances sleep; the first request can take ~1 minute
    MAX_MEDIA_PER_POST=10,
    MAX_MEDIA_MB=200,
    IMAGE_WITH_CAPTION=True,         # also show the caption to Nemotron together with each image (context)
    MEDIA_URL_HOSTS=None,            # e.g. {"res.cloudinary.com"}: only download media from these hosts (None = any https/http host)

)

API_TOKEN = get_secret("EVOLVE_TOKEN", "Backend API token (leave empty if none): ", required=False)
print("Config ready. Backend token:", "set (length %d)" % len(API_TOKEN) if API_TOKEN else "NOT SET: the backend will answer 401")

Config ready. Backend token: set (length 183)


## 2. Install

In [2]:
import subprocess, sys
pkgs = ["transformers", "accelerate", "pillow", "huggingface_hub", "faster-whisper", "soundfile", "requests", "pandas"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=False)
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
GPU available: True Tesla T4


## 3. Hugging Face login

In [3]:
from huggingface_hub import login, notebook_login
_hf = get_secret("HF_TOKEN", required=False)
if _hf:
    login(token=_hf)
    print("Logged in to Hugging Face with HF_TOKEN.")
else:
    notebook_login()          # you must have accepted the Nemotron / Gemma licence on huggingface.co with this account

Logged in to Hugging Face with HF_TOKEN.


## 4. Load all models (one cell)
Nemotron (safety), Whisper (speech) and CLIP (frame triage). The first run validates float16 vs bfloat16 once and saves the result.

In [4]:
# ===== ALL models are loaded in THIS cell: Nemotron (safety), Whisper (speech), CLIP (frame triage) =====
import os, sys, gc, re, glob, json, time, shutil, subprocess, tempfile, unicodedata, warnings, ctypes, site, threading
import numpy as np, torch
from contextlib import contextmanager
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from transformers.utils import logging as hf_logging
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then restart."

safety_proc, safety_model, CAL, MODEL_INFO = None, None, {}, {}
PLACEHOLDER = "What is happening in this image?"

# ---------- Nemotron: loader + batched classifier ----------
def load_safety(dtype, attn="sdpa"):
    global safety_proc, safety_model
    safety_model = None; gc.collect(); torch.cuda.empty_cache()
    if safety_proc is None:
        safety_proc = AutoProcessor.from_pretrained(CFG.SAFETY_ID)
        try:
            safety_proc.tokenizer.padding_side = "left"          # needed for batched generation
        except Exception:
            pass
    kw = dict(torch_dtype=dtype, device_map="auto")
    try:
        m = Gemma3ForConditionalGeneration.from_pretrained(CFG.SAFETY_ID, attn_implementation=attn, **kw)
    except Exception:
        m = Gemma3ForConditionalGeneration.from_pretrained(CFG.SAFETY_ID, **kw)
    safety_model = m.eval()
    print(f"  Nemotron loaded as {next(safety_model.parameters()).dtype} + {attn} | GPU used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

def _gen(batch):
    """batch = [(text|None, image_path|None), ...] -> [raw output string, ...]"""
    convs = []
    for text, img in batch:
        content = []
        if img:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": text if text else PLACEHOLDER})
        convs.append([{"role": "user", "content": content}])
    inputs = safety_proc.apply_chat_template(
        convs[0] if len(convs) == 1 else convs,
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
        padding=True, request_categories="/categories").to(safety_model.device)
    with torch.inference_mode():
        out = safety_model.generate(**inputs, max_new_tokens=CFG.SAFETY_MAX_NEW, do_sample=False)
    n = inputs["input_ids"].shape[1]
    return [safety_proc.decode(o[n:], skip_special_tokens=True).strip() for o in out]

def parse_safety_output(raw):
    """-> {'label': safe|unsafe|review, 'categories': [...], 'raw': ...}   review = could not parse"""
    label, categories = "review", []
    m = re.search(r"User Safety:\s*(safe|unsafe)", raw, re.IGNORECASE)
    if m:
        label = m.group(1).lower()
    c = re.search(r"Safety Categories:\s*(.+)", raw, re.IGNORECASE)
    if c:
        categories = [x.strip() for x in c.group(1).split(",") if x.strip()]
    return {"label": label, "categories": categories, "raw": raw}

def _gen_single_safe(item):
    try:
        return _gen([item])[0]
    except Exception as e:
        print(f"Nemotron failed on one item: {type(e).__name__}: {str(e)[:120]}")
        return ""

def classify_many(items, bs=None):
    """items = [(text|None, image_path|None), ...] -> [parsed result, ...] (same order).
    On out-of-memory the batch is halved; on any other batch error it falls back to one-by-one."""
    bs = bs or CFG.CLASSIFY_BATCH
    results, i = [], 0
    while i < len(items):
        batch = items[i:i + bs]
        try:
            raws = _gen(batch)
        except torch.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if bs > 1:
                bs = max(1, bs // 2); continue
            raws = [""] * len(batch)
        except Exception as e:
            print(f"Batch failed ({type(e).__name__}); retrying one by one")
            raws = [_gen_single_safe(b) for b in batch]
        results += [parse_safety_output(r) for r in raws]
        i += len(batch)
    return results

# ---------- precision: pick the fastest CORRECT one (validated once, then remembered) ----------
VALIDATION = [   # expected labels = what Nemotron returned in earlier runs
    ("I love hiking in the mountains on weekends.", "safe"),
    ("I am going to find you and hurt you badly.", "unsafe"),
    ("आज मौसम बहुत अच्छा है, चलो घूमने चलते हैं।", "safe"),
    ("मैं तुम्हें ढूंढकर मार डालूंगा।", "unsafe"),
    ("kal ka match bahut mast tha, maza aa gaya", "safe"),
    ("main tujhe dhoondh ke maar dunga, bach ke rehna", "unsafe"),
]

def _test_image(path="/tmp/_val.jpg"):
    from PIL import Image
    row = np.linspace(0, 255, 448, dtype=np.uint8)
    Image.fromarray(np.stack([np.tile(row, (448, 1))] * 3, -1)).save(path)
    return path

def validate_safety():
    ok = True
    t0 = time.perf_counter()
    res = classify_many([(t, None) for t, _ in VALIDATION], bs=1)
    t_text = (time.perf_counter() - t0) / len(VALIDATION)
    for (t, exp), r in zip(VALIDATION, res):
        if r["label"] != exp:
            ok = False; print(f"   mismatch: {t[:40]!r} expected {exp}, got {r['label']} ({r['raw'][:50]!r})")
    img = _test_image()
    t0 = time.perf_counter()
    r1 = classify_many([(None, img)], bs=1)[0]; classify_many([(None, img)], bs=1)
    t_img = (time.perf_counter() - t0) / 2
    if r1["label"] == "review":
        ok = False; print("   image output unreadable:", r1["raw"][:80])
    return ok, t_text, t_img

def _sanity_ok():
    res = classify_many([(VALIDATION[0][0], None), (VALIDATION[1][0], None)], bs=1)
    return [r["label"] for r in res] == ["safe", "unsafe"]

def _dname(dt):
    return str(dt).split(".")[-1]

def select_safety_precision():
    gpu, cap = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
    saved = {}
    try:
        if CFG.MODEL_CACHE_FILE and os.path.exists(CFG.MODEL_CACHE_FILE):
            saved = json.load(open(CFG.MODEL_CACHE_FILE))
    except Exception:
        saved = {}
    if saved and saved.get("gpu") == gpu and saved.get("safety_id") == CFG.SAFETY_ID and saved.get("requested") == CFG.SAFETY_DTYPE:
        load_safety(getattr(torch, saved["dtype"]), saved["attn"]); classify_many([("warm up", None)])
        if _sanity_ok():
            CAL.update(text_call_s=saved["text_call_s"], image_call_s=saved["image_call_s"])
            return saved["dtype"], saved["attn"], "saved config"
        print("  saved precision failed the sanity check -> validating again")

    print(f"  GPU {gpu}, compute capability {cap}, bf16 supported = {torch.cuda.is_bf16_supported()} (emulated on T4 = slow)")
    want = {"float16": torch.float16, "bfloat16": torch.bfloat16}.get(CFG.SAFETY_DTYPE) \
           or (torch.bfloat16 if cap[0] >= 8 else torch.float16)
    attempts = [(want, "sdpa")]
    if want == torch.float16 and CFG.TRY_FP16_EAGER:
        attempts.append((torch.float16, "eager"))
    chosen = None
    for dt, attn in attempts:
        load_safety(dt, attn); classify_many([("warm up", None)])
        ok, t_text, t_img = validate_safety()
        print(f"  {_dname(dt):9} {attn:6} answers_match={ok}  {t_text:.2f}s per text call  {t_img:.2f}s per image call")
        if ok:
            chosen = (dt, attn); break
    if chosen is None:
        if attempts[-1] != (torch.bfloat16, "sdpa"):
            print("  float16 failed validation -> using bfloat16 (correct, but slower on T4)")
            load_safety(torch.bfloat16, "sdpa"); classify_many([("warm up", None)])
            ok, t_text, t_img = validate_safety()
            print(f"  bfloat16  sdpa   answers_match={ok}  {t_text:.2f}s per text call  {t_img:.2f}s per image call")
        chosen = (torch.bfloat16, "sdpa") if attempts[-1] != (torch.bfloat16, "sdpa") else attempts[-1]
    CAL.update(text_call_s=t_text, image_call_s=t_img)
    try:
        json.dump({"gpu": gpu, "safety_id": CFG.SAFETY_ID, "requested": CFG.SAFETY_DTYPE, "dtype": _dname(chosen[0]),
                   "attn": chosen[1], "text_call_s": t_text, "image_call_s": t_img}, open(CFG.MODEL_CACHE_FILE, "w"))
    except Exception as e:
        print("  (could not save the precision config:", type(e).__name__, ")")
    return _dname(chosen[0]), chosen[1], "validated now"

# ---------- Whisper: faster-whisper first, transformers fallback ----------
asr, ASR_BACKEND = None, None
ASR_CANDIDATES = ["en", "hi", "gu", "mr", "ur", "pa", "bn", "ta", "te", "kn", "ml"]   # detection is restricted to these

def _preload_cuda12():
    """faster-whisper needs CUDA-12 cuBLAS/cuDNN; install the 12 wheels and load them."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12"], check=False)
    roots = [p for p in site.getsitepackages() + [site.getusersitepackages()] if os.path.isdir(p)]
    for pat in ("nvidia/cublas/lib/libcublasLt.so.12", "nvidia/cublas/lib/libcublas.so.12", "nvidia/cudnn/lib/libcudnn*.so.9"):
        for r in roots:
            for f in sorted(glob.glob(os.path.join(r, pat))):
                try:
                    ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
                except OSError:
                    pass

def _hf_pipe():
    from transformers import pipeline
    base = dict(model=f"openai/whisper-{CFG.ASR_MODEL}", device=0)
    for extra in ({"dtype": torch.float16, "model_kwargs": {"attn_implementation": "sdpa"}},
                  {"torch_dtype": torch.float16, "model_kwargs": {"attn_implementation": "sdpa"}},
                  {"torch_dtype": torch.float16}):
        try:
            return pipeline("automatic-speech-recognition", **base, **extra)
        except Exception:
            continue
    raise RuntimeError("could not build the Whisper pipeline")

def load_asr():
    global asr, ASR_BACKEND
    if asr is not None:
        return
    if CFG.TRY_FASTER_WHISPER:
        try:
            _preload_cuda12()
            from faster_whisper import WhisperModel
            m = WhisperModel(CFG.ASR_MODEL, device="cuda", compute_type="int8_float16")
            list(m.transcribe(np.zeros(32000, dtype=np.float32), beam_size=1)[0])   # surfaces library problems now
            asr, ASR_BACKEND = m, "faster-whisper"
        except Exception as e:
            print("  faster-whisper not usable ->", type(e).__name__, str(e)[:120], "\n  using transformers Whisper instead.")
    if asr is None:
        asr, ASR_BACKEND = _hf_pipe(), "transformers"
        try:
            asr({"raw": np.zeros(32000, dtype=np.float32), "sampling_rate": 16000})
        except Exception:
            pass
    print(f"  ASR ready: whisper-{CFG.ASR_MODEL} via {ASR_BACKEND} | GPU used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

# ---------- CLIP: cheap frame triage ----------
_clip = {}

def load_clip():
    if "model" in _clip or not CFG.TRIAGE:
        return
    try:
        from transformers import CLIPModel, CLIPProcessor
        _clip["proc"] = CLIPProcessor.from_pretrained(CFG.TRIAGE_MODEL)
        _clip["model"] = CLIPModel.from_pretrained(CFG.TRIAGE_MODEL, torch_dtype=torch.float16).to("cuda").eval()
        print("  CLIP triage ready | GPU used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))
    except Exception as e:
        print("  CLIP triage unavailable ->", type(e).__name__, str(e)[:100], "(frames will be evenly spaced instead)")

# ---------- load everything ----------
_t0 = time.perf_counter()
print("1/3 Nemotron (safety)");  _dt, _attn, _how = select_safety_precision()
print("2/3 Whisper (speech)");   load_asr()
print("3/3 CLIP (triage)");      load_clip()
MODEL_INFO.update(safety=CFG.SAFETY_ID, dtype=_dt, attn=_attn, asr=f"whisper-{CFG.ASR_MODEL} ({ASR_BACKEND})",
                  triage=CFG.TRIAGE_MODEL if "model" in _clip else None)
print(f"\nAll models ready in {time.perf_counter() - _t0:.0f}s (precision: {_dt}+{_attn}, {_how}) | "
      f"GPU used: %.1f GB | calibration: {({k: round(v, 2) for k, v in CAL.items()})}" % (torch.cuda.memory_allocated() / 1e9))

1/3 Nemotron (safety)
  GPU Tesla T4, compute capability (7, 5), bf16 supported = True (emulated on T4 = slow)


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/6.60k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/824 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.52k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  CLIP triage ready | GPU used: 8.9 GB

All models ready in 466s (precision: bfloat16+sdpa, validated now) | GPU used: 8.9 GB | calibration: {'text_call_s': 1.96, 'image_call_s': 5.47}


## 5. Helpers

In [5]:
# ===== Helpers: input types, timing, text cleaning, Hindi/Hinglish lexicon, ffmpeg, frames, CLIP triage, ASR reliability =====
NEMO = "Nemotron-3.5 Content Safety"
EXT_MAP = {
    "image": {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff"},
    "video": {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"},
    "audio": {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".aac"},
}

def detect_input_type(text=None, file=None):
    if file:
        if not os.path.exists(file):
            raise FileNotFoundError(f"{os.path.basename(file)} not found")
        ext = os.path.splitext(file)[1].lower()
        return next((k for k, exts in EXT_MAP.items() if ext in exts), "unknown")
    if text and text.strip():
        return "text"
    raise ValueError("Provide text or file.")

class Timer(dict):
    @contextmanager
    def step(self, name):
        t0 = time.perf_counter()
        try:
            yield
        finally:
            self[name] = self.get(name, 0.0) + time.perf_counter() - t0
    def rounded(self):
        return {k: round(v, 2) for k, v in self.items()}

# ---------- text utilities ----------
def chunk_text(text, max_chars=None):
    """Split into pieces of at most max_chars, preferring line breaks."""
    max_chars = max_chars or CFG.MAX_TEXT_CHARS
    chunks, current = [], ""
    for line in text.splitlines():
        while len(line) > max_chars:
            if current:
                chunks.append(current); current = ""
            chunks.append(line[:max_chars]); line = line[max_chars:]
        if len(current) + len(line) + 1 > max_chars:
            if current:
                chunks.append(current)
            current = line
        else:
            current = current + "\n" + line if current else line
    if current.strip():
        chunks.append(current)
    return [c for c in chunks if c.strip()]

def collapse_repeats(text):
    """'I'm not a racist, I'm not a racist, ...' x100 -> one copy. Stops ASR loops from creating fake findings."""
    if len(text) < 40:
        return text
    return re.sub(r"(.{3,80}?)(?:[\s,.!?\u0964-]*\1){2,}", r"\1", text, flags=re.S)

def fmt_t(sec):
    return f"{int(sec // 60)}:{int(sec % 60):02d}"

def group_segments(segs, max_chars=None):
    """ASR segments -> [(label, text)] chunks that keep the start time of each chunk."""
    max_chars = max_chars or CFG.MAX_TEXT_CHARS
    out, cur, size, start = [], [], 0, 0.0
    def flush():
        txt = " ".join(cur)
        for c in chunk_text(txt, max_chars):
            out.append((f"speech {fmt_t(start)}", c))
    for s in segs:
        t = s["text"]
        if cur and size + len(t) + 1 > max_chars:
            flush(); cur, size = [], 0
        if not cur:
            start = s["start"]
        cur.append(t); size += len(t) + 1
    if cur:
        flush()
    return out

# ---------- Hindi / Hinglish lexicon (cheap pre-filter; Nemotron still judges everything) ----------
# STARTER list only. Add your own terms in a file (CFG.LEXICON_FILE) or extend LEXICON_TERMS.
LEXICON_TERMS = [
    "madarchod", "maderchod", "bhenchod", "behenchod", "bhosdike", "bhosdi", "chutiya", "chutiye",
    "gandu", "gaandu", "haramzade", "randi", "lauda", "lund",
    "मादरचोद", "बहनचोद", "भोसड़ीके", "चूतिया", "गांडू", "हरामज़ादे", "हरामजादे", "रंडी", "लौड़ा",
]

def _norm(t):
    t = unicodedata.normalize("NFKC", t).lower()
    return re.sub(r"([a-z\u0900-\u097f])\1+", r"\1", t)        # collapse repeated letters

def build_lexicon():
    terms = list(LEXICON_TERMS)
    if CFG.LEXICON_FILE and os.path.exists(CFG.LEXICON_FILE):
        with open(CFG.LEXICON_FILE, encoding="utf-8") as f:
            terms += [l.strip() for l in f if l.strip() and not l.startswith("#")]
    pats = [re.escape(_norm(t)) for t in terms]
    return re.compile(r"(?<![\w\u0900-\u097f])(?:" + "|".join(pats) + r")(?![\w\u0900-\u097f])")

LEXICON_RE = build_lexicon()

def lexicon_hit(text):
    m = LEXICON_RE.search(_norm(text))
    return m.group(0) if m else None

# ---------- ffmpeg ----------
def _ffprobe(path, args):
    return subprocess.run(["ffprobe", "-v", "error", *args, path], capture_output=True, text=True).stdout.strip()

def duration(path):
    return float(_ffprobe(path, ["-show_entries", "format=duration", "-of", "default=nw=1:nk=1"]) or 0)

def has_audio(path):
    return "audio" in _ffprobe(path, ["-select_streams", "a", "-show_entries", "stream=codec_type", "-of", "csv=p=0"])

def load_audio_np(path, seconds=None):
    """any audio/video file -> float32 mono 16 kHz numpy array (no temp files)."""
    cmd = ["ffmpeg", "-loglevel", "error", "-t", str(seconds or CFG.MAX_SECONDS), "-i", path,
           "-vn", "-ac", "1", "-ar", "16000", "-f", "f32le", "-"]
    return np.frombuffer(subprocess.run(cmd, capture_output=True, check=True).stdout, dtype=np.float32)

def dedupe_frames(files):
    from PIL import Image
    if CFG.FRAME_DEDUPE_DIFF <= 0:
        return list(files)
    kept, last = [], None
    for f in files:
        with Image.open(f) as im:
            sig = np.asarray(im.convert("L").resize((16, 16)), dtype=np.float32)
        d = None if last is None else np.abs(sig - last)
        # keep if the picture changed overall OR any small region changed a lot (e.g. an object appears in a corner)
        if d is None or d.mean() >= CFG.FRAME_DEDUPE_DIFF or d.max() >= 8 * CFG.FRAME_DEDUPE_DIFF:
            kept.append(f); last = sig
    return kept

def plan_frames(dur, with_audio=True):
    """-> (n_send, n_score, span). n_send = frames for the expensive Nemotron call (from the latency budget and the
    MEASURED seconds per call). n_score = frames the cheap CLIP triage looks at (triage on) or the same as n_send (off)."""
    span = min(dur, CFG.MAX_SECONDS) if dur else CFG.MAX_SECONDS
    tiles, per = max(1, CFG.MONTAGE_TILES), CAL.get("image_call_s", 3.0)
    asr_s = (0.15 * span + 2) if with_audio else 0
    calls = max(1, int(((CFG.LATENCY_BUDGET_S or 1e9) - asr_s - 3) / per))     # 3 s reserved for ffmpeg + text checks
    n_send = min(CFG.MAX_FRAMES, calls * tiles)
    cap = int(span / CFG.MIN_FRAME_GAP_S) + 1
    if CFG.TRIAGE and "model" in _clip:
        n_score = max(min(CFG.TRIAGE_FRAMES, cap), min(n_send, cap))
        n_send = min(n_send, n_score)
    else:
        n_send = min(max(n_send, min(CFG.MAX_FRAMES, int(np.ceil(span / CFG.MAX_FRAME_GAP_S)))), cap)   # coverage floor beats the budget
        n_score = n_send
    return max(1, n_send), max(1, n_score), span

def extract_frames(path, out_dir, dur, n_frames, span):
    """Evenly spaced frames over the WHOLE (capped) video -> ([(label, jpg_path)], n_sampled, interval)."""
    interval = max(span / n_frames, 0.1)
    s = CFG.FRAME_SHORT_SIDE
    vf = f"fps=1/{interval:.3f},scale=w='if(gt(iw,ih),-2,{s})':h='if(gt(iw,ih),{s},-2)'"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{interval / 2:.3f}", "-t", str(span), "-i", path,
                    "-an", "-vf", vf, "-q:v", "4", "-frames:v", str(n_frames), f"{out_dir}/frame_%04d.jpg"], check=True)
    files = sorted(glob.glob(f"{out_dir}/frame_*.jpg"))
    if not files:                                                    # very short clip: grab the first frame
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-an", "-frames:v", "1",
                        "-vf", vf.split(",", 1)[1], f"{out_dir}/frame_0001.jpg"], check=True)
        files = sorted(glob.glob(f"{out_dir}/frame_*.jpg"))
    kept = dedupe_frames(files)
    def label(f):
        k = int(re.search(r"(\d+)\.jpg$", f).group(1))
        return f"frame {fmt_t((k - 1) * interval + interval / 2)}"
    return [(label(f), f) for f in kept], len(files), interval

def build_montage(paths, out_path):
    """Pack up to 9 frames into ONE 896x896 image (one Nemotron call instead of several)."""
    from PIL import Image
    cols = 2 if len(paths) <= 4 else 3
    cell = 896 // cols
    canvas = Image.new("RGB", (896, 896), (0, 0, 0))
    for i, p in enumerate(paths):
        with Image.open(p) as im:
            im = im.convert("RGB"); im.thumbnail((cell, cell))
            canvas.paste(im, ((i % cols) * cell + (cell - im.width) // 2, (i // cols) * cell + (cell - im.height) // 2))
    canvas.save(out_path, quality=90)
    return out_path

def mean_gop(path, span):
    """average seconds between keyframes (reads packet headers only: milliseconds, no decoding)"""
    out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-read_intervals", f"0%+{span:.0f}",
                          "-show_entries", "packet=flags", "-of", "csv=p=0", path], capture_output=True, text=True).stdout
    return span / max(1, out.count("K"))

def _clean_frames(out_dir):
    for f in glob.glob(f"{out_dir}/frame_*.jpg"):
        os.remove(f)

def extract_frames_keyframes(path, out_dir, n, span):
    """ONE ffmpeg pass that decodes only keyframes (skips ~95% of the decoding work), sampled with the fps filter."""
    interval = max(span / n, 0.1)
    s = CFG.FRAME_SHORT_SIDE
    vf = f"fps=1/{interval:.3f},scale=w='if(gt(iw,ih),-2,{s})':h='if(gt(iw,ih),{s},-2)'"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-skip_frame", "nokey", "-ss", f"{interval / 2:.3f}", "-t", str(span),
                    "-i", path, "-an", "-vf", vf, "-q:v", "4", "-frames:v", str(n), f"{out_dir}/frame_%04d.jpg"], check=True)
    files = sorted(glob.glob(f"{out_dir}/frame_*.jpg"))
    kept = dedupe_frames(files)
    def label(f):
        k = int(re.search(r"(\d+)[.]jpg$", f).group(1))
        return f"frame {fmt_t((k - 1) * interval + interval / 2)}"
    return [(label(f), f) for f in kept], len(files)

def extract_frames_auto(path, out_dir, dur, n, span):
    """Keyframes-only decoding when the video has frequent keyframes, else the sequential method.
    -> (frames, n_sampled, interval, how)"""
    interval = max(span / n, 0.1)
    if CFG.FAST_SEEK and n > 1:
        try:
            gop = mean_gop(path, span)
            if gop <= 2 * interval:
                frames, n_raw = extract_frames_keyframes(path, out_dir, n, span)
                if frames:
                    return frames, n_raw, interval, f"keyframes-only, keyframe every {gop:.1f}s"
            _clean_frames(out_dir)
        except Exception:
            _clean_frames(out_dir)
    frames, n_raw, interval = extract_frames(path, out_dir, dur, n, span)
    return frames, n_raw, interval, "sequential"

# ---------- frame triage (CLIP) ----------
RISK_PROMPTS = [
    "a photo of a violent fight or a person being attacked",
    "a photo of blood, gore or a dead body",
    "a photo of a person pointing a gun or holding a weapon",
    "a photo of nudity or sexual content",
    "a photo of an explosion, a fire or a terrorist attack",
    "a photo of people using illegal drugs",
    "a photo of a person hurting themselves",
]
SAFE_PROMPTS = [
    "a photo of an ordinary everyday scene", "a photo of people talking", "a photo of nature or a landscape",
    "a screenshot of a computer screen or code", "a photo of a car or a vehicle", "a cartoon or an animation",
    "a photo of food", "a photo of a sports game", "a movie scene with actors",
]

def clip_risk(paths):
    """-> list of risk scores in [0,1] (higher = look first), or None if triage is unavailable (caller falls back to even spacing)"""
    if "model" not in _clip:
        return None
    try:
        from PIL import Image
        imgs = []
        for p in paths:
            with Image.open(p) as im:
                imgs.append(im.convert("RGB").copy())
        x = _clip["proc"](text=RISK_PROMPTS + SAFE_PROMPTS, images=imgs, return_tensors="pt", padding=True).to("cuda")
        with torch.inference_mode():
            logits = _clip["model"](input_ids=x["input_ids"], attention_mask=x["attention_mask"],
                                    pixel_values=x["pixel_values"].half()).logits_per_image.float()
        return logits.softmax(-1)[:, :len(RISK_PROMPTS)].max(-1).values.tolist()
    except Exception as e:
        print("frame triage unavailable ->", type(e).__name__, str(e)[:100])
        return None

def select_frames(frames, scores, n_send):
    """Riskiest frame of each of n_send equal time segments (keeps full coverage over time),
    plus any frame whose risk >= TRIAGE_ALWAYS_SEND. Without scores: evenly spaced subset."""
    n = len(frames)
    if n <= n_send:
        return list(frames)
    if scores is None:
        idx = sorted(set(np.linspace(0, n - 1, n_send).round().astype(int).tolist()))
        return [frames[i] for i in idx]
    chosen, bounds = set(), np.linspace(0, n, n_send + 1).round().astype(int)
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        if hi > lo:
            chosen.add(lo + int(np.argmax(scores[lo:hi])))
    extra = [i for i in np.argsort(scores)[::-1] if scores[i] >= CFG.TRIAGE_ALWAYS_SEND and i not in chosen]
    chosen.update(int(i) for i in extra[:CFG.TRIAGE_ALWAYS_MAX])
    return [frames[i] for i in sorted(chosen)]

# ---------- ASR reliability helpers ----------
def loop_coverage(text, n=5):
    """share of the text covered by the most repeated n-word phrase (0 = no loop, 1 = pure loop)"""
    from collections import Counter
    w = re.sub(r"[^\w\u0900-\u097F\s]", " ", text.lower()).split()
    if len(w) < n * 4:
        return 0.0
    sh = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    cnt = Counter(sh)
    if cnt.most_common(1)[0][1] < 4:
        return 0.0
    covered = set()
    for i, g in enumerate(sh):
        if cnt[g] >= 3:
            covered.update(range(i, i + n))
    return len(covered) / len(w)

def is_looping(text):
    return loop_coverage(text) >= 0.25

def dedupe_similar_sentences(text, thr=0.8):
    """keep a sentence only if it is not (almost) a repeat of one already kept"""
    kept, sets = [], []
    for sent in re.split(r"(?<=[.!?\u0964])\s+", text):
        ws = set(re.sub(r"[^\w\u0900-\u097F\s]", " ", sent.lower()).split())
        if not ws:
            continue
        if any(len(ws & k) / len(ws | k) >= thr for k in sets):
            continue
        kept.append(sent); sets.append(ws)
    return " ".join(kept)

def is_junk(text):
    """'.'  'hmm hmm' etc.: Whisper hallucinations on music/noise"""
    letters = [c for c in text if c.isalpha()]
    return len(letters) < 3 or (len(set(letters)) <= 2 and len(letters) < 30)

def is_silent(samples):
    if len(samples) < 1600:
        return True
    n = len(samples) // 320 * 320
    rms = np.sqrt((samples[:n].reshape(-1, 320) ** 2).mean(1))
    return float(np.percentile(rms, 95)) < 0.003

## 6. Audio branch

In [6]:
# ===== Audio branch: language detection + Whisper transcription =====
def detect_language(samples):
    """Restricted to ASR_CANDIDATES, so Hindi is not mistaken for Arabic / Welsh / etc. -> (lang, prob) or (None, None)"""
    try:
        x = samples[:16000 * 30]
        if ASR_BACKEND == "faster-whisper":
            _, _, allp = asr.detect_language(x)
            d = {l: p for l, p in allp if l in ASR_CANDIDATES}
            tot = sum(d.values()) or 1.0
            d = {l: p / tot for l, p in d.items()}
        else:
            tok, model = asr.tokenizer, asr.model
            feats = asr.feature_extractor(x, sampling_rate=16000, return_tensors="pt").input_features.to(model.device, model.dtype)
            sot = tok.convert_tokens_to_ids("<|startoftranscript|>")
            with torch.inference_mode():
                logits = model(input_features=feats, decoder_input_ids=torch.tensor([[sot]], device=model.device)).logits[0, -1].float()
            ids = {l: tok.convert_tokens_to_ids(f"<|{l}|>") for l in ASR_CANDIDATES}
            ids = {l: i for l, i in ids.items() if i is not None and i != tok.unk_token_id}
            d = dict(zip(ids.keys(), torch.softmax(logits[list(ids.values())], -1).tolist()))
        best = max(d, key=d.get)
        return best, d[best]
    except Exception as e:
        print("language detection skipped:", type(e).__name__, str(e)[:80])
        return None, None

def _run_asr(samples, lang, task="transcribe"):
    if ASR_BACKEND == "faster-whisper":
        it, _ = asr.transcribe(samples, language=lang, task=task, beam_size=CFG.ASR_BEAM, vad_filter=True,
                               vad_parameters=dict(min_silence_duration_ms=500), condition_on_previous_text=False,
                               compression_ratio_threshold=2.4, no_speech_threshold=0.6)
        return [dict(start=s.start, end=s.end, text=s.text.strip()) for s in it]
    gk = {"task": task}
    if lang:
        gk["language"] = lang
    out = asr({"raw": samples, "sampling_rate": 16000}, chunk_length_s=30, batch_size=16,
              return_timestamps=True, generate_kwargs=gk)
    segs = []
    for c in out.get("chunks", []):
        ts = c.get("timestamp") or (0, 0)
        segs.append(dict(start=ts[0] or 0, end=ts[1] or 0, text=c["text"].strip()))
    if not segs and out.get("text", "").strip():
        segs = [dict(start=0, end=0, text=out["text"].strip())]
    return segs

def transcribe(samples):
    """float32 16 kHz array -> segments + language + reliability flag"""
    res = {"segments": [], "language": None, "language_prob": None, "unreliable": False, "note": ""}
    if is_silent(samples):
        res["note"] = "silent"                                        # skip ASR completely: saves time
        return res
    lang, p = (CFG.ASR_LANGUAGE, 1.0) if CFG.ASR_LANGUAGE else detect_language(samples)
    segs = _run_asr(samples, lang)
    text = " ".join(s["text"] for s in segs)
    if is_looping(text):                                              # Whisper got stuck
        if lang != "hi":
            segs2 = _run_asr(samples, "hi")
            text2 = " ".join(s["text"] for s in segs2)
            if not is_looping(text2):
                segs, text, lang, res["note"] = segs2, text2, "hi", "retried as Hindi"
        if is_looping(text):
            res["unreliable"] = loop_coverage(text) >= 0.5
            clean = dedupe_similar_sentences(collapse_repeats(text))
            segs = [dict(start=segs[0]["start"], end=segs[-1]["end"], text=clean)] if segs else []
    for s in segs:
        s["text"] = collapse_repeats(s["text"])
    res["segments"] = [s for s in segs if s["text"].strip() and not is_junk(s["text"])]
    res["language"], res["language_prob"] = lang, p
    return res

def audio_branch(path):
    """runs in a worker thread: file -> transcript. Returns dict with its own timings."""
    t = {}
    t0 = time.perf_counter(); samples = load_audio_np(path); t["audio_load"] = time.perf_counter() - t0
    t0 = time.perf_counter(); tr = transcribe(samples); t["asr"] = time.perf_counter() - t0
    return {"transcript": tr, "timings": t, "seconds": len(samples) / 16000}

## 7. Decision core
`decide()` is the only place that makes SAFE / REVIEW / UNSAFE; every result carries reason codes.

In [7]:
# ===== Decision core: every check produces a finding; decide() is the only place that makes SAFE / REVIEW / UNSAFE =====
def _finding(kind, where, source, cats=(), code=None, path=None, note=""):
    return {"kind": kind, "where": where, "source": source, "cats": list(cats),
            "code": code, "path": path, "note": note}

def _from_results(results, names, source, paths=None):
    out = []
    for i, (name, r) in enumerate(zip(names, results)):
        path = paths[i] if paths else None
        if r["label"] == "unsafe":
            cats = r["categories"]
            soft = bool(cats) and all(c.lower() in CFG.SOFT_CATEGORIES for c in cats)
            out.append(_finding("soft" if soft else "unsafe", name, source, cats, path=path))
        elif r["label"] == "review":
            out.append(_finding("parse", name, source, code="PARSE_FAIL", path=path, note=r["raw"][:80]))
    return out

def classify_until(items, names, source, need=1, paths=None):
    """Classify in batches and STOP as soon as `need` unsafe findings exist (early stop = speed)."""
    findings, n_unsafe, bs = [], 0, CFG.CLASSIFY_BATCH
    for i in range(0, len(items), bs):
        res = classify_many(items[i:i + bs], bs)
        fs = _from_results(res, names[i:i + bs], source, paths[i:i + bs] if paths else None)
        findings += fs
        n_unsafe += sum(f["kind"] == "unsafe" for f in fs)
        if n_unsafe >= need:
            break
    return findings

def _code(f):
    if f["code"]:
        return f["code"]
    return {"unsafe": "FLAGGED_", "soft": "SOFT_"}.get(f["kind"], "ERROR_") + f["source"].upper()

def decide(findings, partial=False):
    unsafe = [f for f in findings if f["kind"] == "unsafe"]
    frame_unsafe = [f for f in unsafe if f["source"] == "frame"]
    if frame_unsafe and len(frame_unsafe) == len(unsafe) and len(frame_unsafe) < CFG.MIN_FLAGGED_FRAMES:
        for f in frame_unsafe:
            f["kind"], f["code"] = "soft", "FRAME_VOTES_LOW"
        unsafe = []
    cats = sorted({c for f in findings if f["kind"] in ("unsafe", "soft") for c in f["cats"]})
    codes = sorted({_code(f) for f in findings} | ({"PARTIAL_INPUT"} if partial else set()))
    if unsafe:
        why = "; ".join(f"{f['where']} ({', '.join(f['cats']) or 'no category'})" for f in unsafe[:5])
        return "UNSAFE", cats, codes, "Flagged in: " + why
    if findings or partial:
        why = [f"{f['where']}: {_code(f)}" + (f" ({', '.join(f['cats'])})" if f["cats"] else "") for f in findings[:5]]
        if partial:
            why.append(f"input longer than limits, only the first {CFG.MAX_SECONDS}s / {CFG.MAX_CHUNKS} text chunks were checked")
        return "REVIEW", cats, codes, "Needs a human look: " + "; ".join(why)
    return "SAFE", [], codes, "No unsafe content found."

def _result(decision, cats, codes, explanation, input_type, models, timings, details=None):
    return {"decision": decision, "categories": cats, "reasons": codes, "explanation": explanation,
            "input_type": input_type, "models_used": models,
            "timings": timings.rounded() if hasattr(timings, "rounded") else timings, "details": details or {}}

def classify_frames_montage(frames, T):
    """Pass 1: pack CFG.MONTAGE_TILES frames per image -> ONE Nemotron call per group (fast).
    Pass 2: only for a flagged group, re-check its frames one by one (confirms, locates, removes montage-only false alarms).
    A group flagged but not confirmed frame-by-frame -> soft finding (REVIEW), never UNSAFE."""
    k = max(2, CFG.MONTAGE_TILES)
    groups = [frames[i:i + k] for i in range(0, len(frames), k)]
    out_dir = os.path.dirname(frames[0][1])
    with T.step("montage_build"):
        paths = [g[0][1] if len(g) == 1 else build_montage([p for _, p in g], f"{out_dir}/montage_{gi:03d}.jpg")
                 for gi, g in enumerate(groups)]
    findings, confirmed, need, bs = [], 0, CFG.MIN_FLAGGED_FRAMES, CFG.CLASSIFY_BATCH
    for i in range(0, len(groups), bs):
        res = classify_many([(None, p) for p in paths[i:i + bs]], bs)
        for j, r in enumerate(res):
            g = groups[i + j]
            if r["label"] == "safe":
                continue
            if len(g) == 1:                                       # already a single frame
                fs = _from_results([r], [g[0][0]], "frame", [g[0][1]])
                findings += fs; confirmed += sum(f["kind"] == "unsafe" for f in fs)
            else:
                fs = classify_until([(None, p) for _, p in g], [n for n, _ in g], "frame",
                                    need=max(1, need - confirmed), paths=[p for _, p in g])
                findings += fs; confirmed += sum(f["kind"] == "unsafe" for f in fs)
                if not fs:
                    findings.append(_finding("soft", f"{g[0][0]} to {g[-1][0].replace('frame ', '')}", "frame",
                                             r["categories"], code="MONTAGE_UNCONFIRMED", path=paths[i + j]))
            if confirmed >= need:
                return findings
    return findings

def run_checks(input_type, models, T, text_items=(), image_items=(), image_caption=None,
               findings=None, partial=False, details=None):
    """text_items  = [(name, text)]  -> lexicon, then Nemotron (batched, early stop)
       image_items = [(name, path)]  -> Nemotron vision (batched, early stop)
    Cheapest first, and we stop as soon as something is UNSAFE."""
    findings, details = list(findings or []), dict(details or {})

    with T.step("lexicon"):                                           # 1) lexicon on all text (microseconds)
        for name, text in text_items:
            hit = text and lexicon_hit(text)
            if hit:
                kind = "unsafe" if CFG.LEXICON_ACTION.upper() == "UNSAFE" else "soft"
                findings.append(_finding(kind, name, "lexicon", ["Profanity / abuse (lexicon)"], code=f"LEXICON_{kind.upper()}"))
    hard = any(f["kind"] == "unsafe" for f in findings)

    if text_items and not hard:                                       # 2) Nemotron on text chunks
        chunks, names = [], []
        for name, text in text_items:
            if not text or not text.strip():
                continue
            for k, c in enumerate(chunk_text(text)):
                chunks.append(c); names.append(name if k == 0 else f"{name} #{k + 1}")
        if len(chunks) > CFG.MAX_CHUNKS:
            chunks, names, partial = chunks[:CFG.MAX_CHUNKS], names[:CFG.MAX_CHUNKS], True
        with T.step("nemotron_text"):
            findings += classify_until([(c, None) for c in chunks], names, "text")
        hard = any(f["kind"] == "unsafe" for f in findings)

    if image_items and not hard:                                      # 3) Nemotron on images / frames
        names = [n for n, _ in image_items]; paths = [p for _, p in image_items]
        src = "frame" if input_type == "video" else "image"
        need = CFG.MIN_FLAGGED_FRAMES if src == "frame" else 1
        with T.step("nemotron_images"):
            if src == "frame" and CFG.MONTAGE_TILES > 1 and len(image_items) > 1:
                findings += classify_frames_montage(image_items, T)
            else:
                findings += classify_until([(image_caption, p) for p in paths], names, src, need=need, paths=paths)

    decision, cats, codes, why = decide(findings, partial)
    return _result(decision, cats, codes, why, input_type, models, T, details)

## 8. `moderate()` for one input, and warm-up

In [8]:
# ===== moderate(): one function per input type, plus the dispatcher and the warm-up =====
ASR_NAME = lambda: f"whisper-{CFG.ASR_MODEL} ({ASR_BACKEND})"

def moderate_text(text):
    T = Timer()
    return run_checks("text", [NEMO], T, text_items=[("text", text)])

def moderate_image(path, caption=None):
    T = Timer()
    return run_checks("image", [NEMO], T, image_items=[("image", path)], image_caption=caption)

def _speech_items(audio, details):
    """audio_branch result -> (text_items, findings); fills details"""
    tr, items, findings = audio["transcript"], [], []
    details["language"] = tr["language"]
    details["language_prob"] = None if tr["language_prob"] is None else round(tr["language_prob"], 2)
    if not tr["segments"]:
        details["speech"] = "silent" if tr.get("note") == "silent" else "no speech detected"
    else:
        items += group_segments(tr["segments"])
        details["transcript"] = " ".join(s["text"] for s in tr["segments"])[:3000]
    if tr.get("unreliable"):
        findings.append(_finding("soft", "speech", "asr", code="ASR_UNRELIABLE",
                                 note="speech-to-text got stuck in a loop; transcript cannot be trusted"))
    if tr.get("note"):
        details["asr_note"] = tr["note"]
    return items, findings

def moderate_audio(path, caption=None):
    T, details = Timer(), {}
    findings, items = [], []
    try:
        audio = audio_branch(path)
        for k, v in audio["timings"].items():
            T[k] = v
        items, findings = _speech_items(audio, details)
        partial = duration(path) > CFG.MAX_SECONDS
    except Exception as e:
        partial = False
        findings.append(_finding("error", "audio", "audio", code="STAGE_ERROR", note=f"{type(e).__name__}: {e}"))
    if caption:
        items.append(("caption", caption))
    return run_checks("audio", [ASR_NAME(), NEMO], T, text_items=items, findings=findings, partial=partial, details=details)

def moderate_video(path, caption=None):
    T, details = Timer(), {}
    findings, items, frames = [], [], []
    dur = duration(path)
    partial = dur > CFG.MAX_SECONDS
    with tempfile.TemporaryDirectory() as tmp:
        with ThreadPoolExecutor(max_workers=1) as ex:
            audio_on = has_audio(path)
            fut = ex.submit(audio_branch, path) if audio_on else None             # GPU ASR runs while CPU extracts frames
            try:
                n_send, n_score, span = plan_frames(dur, audio_on)
                with T.step("frames_extract"):
                    all_frames, n_raw, interval, how = extract_frames_auto(path, tmp, dur, n_score, span)
                scores = None
                if CFG.TRIAGE and len(all_frames) > n_send:
                    with T.step("triage"):
                        scores = clip_risk([p for _, p in all_frames])
                frames = select_frames(all_frames, scores, n_send)
                calls = -(-len(frames) // max(1, CFG.MONTAGE_TILES))
                details["frames"] = (f"{n_raw} sampled over {span:.0f}s ({how}), {len(all_frames)} distinct"
                                     + (", ranked by CLIP" if scores is not None else "")
                                     + f" -> {len(frames)} sent to Nemotron in about {calls} call(s)")
                if scores is not None:
                    details["top_risk"] = ", ".join(f"{all_frames[j][0]}={scores[j]:.2f}"
                                                    for j in np.argsort(scores)[::-1][:3])
            except Exception as e:
                findings.append(_finding("error", "video", "frame", code="STAGE_ERROR", note=f"{type(e).__name__}: {e}"))
            if fut is not None:
                try:
                    audio = fut.result()
                    for k, v in audio["timings"].items():
                        T[k] = v
                    items, fs = _speech_items(audio, details); findings += fs
                except Exception as e:
                    findings.append(_finding("error", "audio track", "audio", code="STAGE_ERROR", note=f"{type(e).__name__}: {e}"))
            else:
                details["speech"] = "video has no audio track"
        if caption:
            items.append(("caption", caption))
        # must stay inside the `with` so the frame files still exist for Nemotron
        return run_checks("video", [ASR_NAME(), NEMO], T, text_items=items, image_items=frames,
                          findings=findings, partial=partial, details=details)

def moderate(text=None, file=None, caption=None):
    """moderate(text="...")   or   moderate(file="/path/x.mp4", caption="optional")     # image, video, audio
    `text` is ALWAYS treated as plain text, never as a file path (important on a server)."""
    if file and text and caption is None:
        caption, text = text, None
    t0, kind = time.perf_counter(), "unknown"
    try:
        kind = detect_input_type(text=None if file else text, file=file)
        if kind == "text":
            r = moderate_text(text)
        elif kind == "image":
            r = moderate_image(file, caption)
        elif kind == "video":
            r = moderate_video(file, caption)
        elif kind == "audio":
            r = moderate_audio(file, caption)
        else:
            r = _result("REVIEW", [], ["UNSUPPORTED_TYPE"], f"Unsupported file type: {os.path.splitext(file or '')[1]}", "unknown", [], Timer())
    except Exception as e:                                   # fail closed: never SAFE when something broke
        r = _result("REVIEW", [], ["STAGE_ERROR"], f"Processing failed: {type(e).__name__}: {e}", kind, [], Timer())
    r["timings"]["total"] = round(time.perf_counter() - t0, 2)
    return r

def show(r, details=False):
    tm = r["timings"]
    print(f"[{r['decision']:6}] {r['input_type']}  |  total {tm.get('total', 0)}s")
    if r["categories"]:
        print("  categories :", ", ".join(r["categories"]))
    print("  why        :", r["explanation"])
    print("  reason code:", ", ".join(r["reasons"]) or "-")
    print("  time       :", "  ".join(f"{k} {v}s" for k, v in tm.items() if k != "total" and v >= 0.05))
    if details:
        for k, v in r["details"].items():
            print(f"  [{k}] {str(v)[:400]}")

def warm_up():
    """Pays the one-time GPU start-up costs before the first real request (first video otherwise ~5 s slower)."""
    t0 = time.perf_counter()
    noise = (0.05 * np.random.default_rng(0).standard_normal(16000 * 6)).astype(np.float32)
    try:
        transcribe(noise)
    except Exception as e:
        print("ASR warm-up skipped:", type(e).__name__)
    img = _test_image()
    if "model" in _clip:
        clip_risk([img, img])
    classify_many([(None, img)], bs=1)                        # image-shaped Nemotron call (same size as a montage)
    print(f"Warm-up done in {time.perf_counter() - t0:.1f}s | GPU used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

warm_up()

Warm-up done in 6.0s | GPU used: 8.9 GB


## 9. Backend client: one post becomes one decision
Caption is checked once, every media file is downloaded and checked, the worst item wins.

In [9]:
# ===== Platform client + post-level moderation: fetch a post, check caption + every media file, merge into ONE decision =====
import requests, tempfile
from urllib.parse import urlparse, urljoin

GPU_LOCK = threading.Lock()          # one GPU = one model call at a time (worker, /moderate and /moderate/text all share it)
RANK = {"SAFE": 0, "REVIEW": 1, "UNSAFE": 2}

class PostGone(Exception):
    """the post was deleted: nothing to moderate, no retry"""
class MediaTooLarge(Exception):
    pass

def _headers():
    return {"Authorization": f"Bearer {API_TOKEN}"} if API_TOKEN else {}

def _first(d, keys, default=""):
    for k in keys:
        v = d.get(k)
        if v not in (None, ""):
            return v
    return default

def fetch_user_data(user_id):
    url = CFG.API_BASE.rstrip("/") + CFG.POSTS_ENDPOINT.format(user_id=user_id)
    r = requests.get(url, headers=_headers(), timeout=CFG.REQUEST_TIMEOUT)
    r.raise_for_status()
    data = r.json()
    return data.get("data", data) if isinstance(data, dict) else {}

def check_backend_auth(user_id=1):
    """Run this once after setting EVOLVE_TOKEN: tells you at once whether the backend accepts it."""
    print("token set:", bool(API_TOKEN), "| length:", len(API_TOKEN or ""))
    try:
        d = fetch_user_data(user_id)
        print(f"OK: backend accepted the request, {len(d.get('posts', []))} post(s) for user {user_id}")
        return True
    except requests.HTTPError as e:
        print(f"FAILED: HTTP {e.response.status_code}", "-> the token is missing, wrong or expired" if e.response.status_code in (401, 403) else "")
    except Exception as e:
        print("FAILED:", type(e).__name__, str(e)[:150])
    return False

def normalize_post(p, i=0):
    """backend post JSON -> {id, text, type, deleted, media:[{id, url, type}]}  (caption + title = text; media in order)"""
    text = "\n".join(x for x in [_first(p, ["title"]), _first(p, ["caption", "content", "text", "body", "description"])] if x)
    media = [{"id": m.get("id"), "url": _first(m, ["mediaUrl", "media_url", "url"], None), "type": m.get("mediaType")}
             for m in sorted(p.get("media") or [], key=lambda m: m.get("position") or 0)]
    return {"id": p.get("id", i), "text": text, "type": p.get("postType"), "deleted": bool(p.get("deletedAt")),
            "media": [m for m in media if m["url"]]}

def find_post(user_id, post_id):
    data = fetch_user_data(user_id)
    for i, p in enumerate(data.get("posts", [])):
        if str(p.get("id")) == str(post_id):
            return normalize_post(p, i)
    return None

_CT_EXT = {"image/jpeg": ".jpg", "image/png": ".png", "image/webp": ".webp", "image/gif": ".gif", "image/bmp": ".bmp",
           "video/mp4": ".mp4", "video/quicktime": ".mov", "video/webm": ".webm", "video/x-matroska": ".mkv",
           "audio/mpeg": ".mp3", "audio/mp4": ".m4a", "audio/wav": ".wav", "audio/x-wav": ".wav", "audio/ogg": ".ogg",
           "audio/flac": ".flac", "audio/aac": ".aac"}
_ALL_EXT = set().union(*EXT_MAP.values())
_TYPE_DEFAULT = {"IMAGE": ".jpg", "VIDEO": ".mp4", "AUDIO": ".mp3"}

def get_media(m, post_id, dest_dir):
    """Download one media item into dest_dir (size-capped). The login token goes ONLY to our own backend."""
    url = m["url"]
    if not url.startswith("http"):
        url = urljoin(CFG.API_BASE.rstrip("/") + "/", url.lstrip("/"))
    host = urlparse(url).netloc
    if CFG.MEDIA_URL_HOSTS and host not in CFG.MEDIA_URL_HOSTS and host != urlparse(CFG.API_BASE).netloc:
        raise ValueError(f"media host not allowed: {host}")
    limit = CFG.MAX_MEDIA_MB * 1024 * 1024
    headers = _headers() if host == urlparse(CFG.API_BASE).netloc else {}
    with requests.get(url, headers=headers, stream=True, timeout=CFG.REQUEST_TIMEOUT) as r:
        r.raise_for_status()
        if int(r.headers.get("Content-Length") or 0) > limit:
            raise MediaTooLarge(f"larger than {CFG.MAX_MEDIA_MB} MB")
        ext = os.path.splitext(urlparse(url).path)[1].lower()
        if ext not in _ALL_EXT:
            ext = _CT_EXT.get((r.headers.get("Content-Type") or "").split(";")[0].strip().lower()) \
                  or _TYPE_DEFAULT.get((m.get("type") or "").upper(), "")
        safe = re.sub(r"[^\w.-]", "_", f"post_{post_id}_media_{m.get('id')}")
        path = os.path.join(dest_dir, safe + ext)
        size = 0
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                size += len(chunk)
                if size > limit:
                    raise MediaTooLarge(f"larger than {CFG.MAX_MEDIA_MB} MB")
                f.write(chunk)
    return path

# ---------- one post -> one decision ----------
def _item(name, kind, decision, cats=(), reasons=(), why="", seconds=0.0, details=None):
    return {"item": name, "type": kind, "decision": decision, "categories": list(cats), "reasons": list(reasons),
            "explanation": why, "seconds": round(seconds, 2), "details": details if decision != "SAFE" else {}}

def _run_item(name, kind, fn):
    """run one moderate() call under the GPU lock; a crash becomes a REVIEW item, never a silent SAFE"""
    t0 = time.perf_counter()
    try:
        with GPU_LOCK:
            r = fn()
        return _item(name, r["input_type"], r["decision"], r["categories"], r["reasons"], r["explanation"],
                     time.perf_counter() - t0, r.get("details"))
    except Exception as e:
        return _item(name, kind, "REVIEW", [], ["STAGE_ERROR"], f"{type(e).__name__}: {e}", time.perf_counter() - t0)

def merge_items(items):
    """worst item wins: UNSAFE > REVIEW > SAFE"""
    decision = max((i["decision"] for i in items), key=RANK.get)
    cats = sorted({c for i in items for c in i["categories"]})
    reasons = sorted({c for i in items if i["decision"] != "SAFE" for c in i["reasons"]})
    bad = sorted((i for i in items if i["decision"] != "SAFE"), key=lambda i: -RANK[i["decision"]])
    why = ("; ".join(f"{i['item']}: {i['decision']} ({i['explanation']})" for i in bad[:5])
           if bad else f"No unsafe content found in {len(items)} item(s).")
    return decision, cats, reasons, why

def moderate_post(post):
    t0, items = time.perf_counter(), []
    text = (post.get("text") or "").strip()
    media = post.get("media") or []

    if text:
        items.append(_run_item("caption", "text", lambda: moderate(text=text)))
    with tempfile.TemporaryDirectory() as tmp:                         # media files are deleted as soon as the post is done
        for k, m in enumerate(media[:CFG.MAX_MEDIA_PER_POST]):
            name = f"media {k + 1}"
            try:
                path = get_media(m, post["id"], tmp)
            except MediaTooLarge as e:
                items.append(_item(name, (m.get("type") or "media").lower(), "REVIEW", [], ["MEDIA_TOO_LARGE"], str(e))); continue
            except Exception as e:
                items.append(_item(name, (m.get("type") or "media").lower(), "REVIEW", [], ["MEDIA_DOWNLOAD_FAILED"], f"{type(e).__name__}: {str(e)[:120]}")); continue
            is_image = os.path.splitext(path)[1].lower() in EXT_MAP["image"]
            cap = text if (is_image and text and CFG.IMAGE_WITH_CAPTION) else None
            items.append(_run_item(name, (m.get("type") or "media").lower(), lambda p=path, c=cap: moderate(file=p, caption=c)))
    if len(media) > CFG.MAX_MEDIA_PER_POST:
        items.append(_item("media", "media", "REVIEW", [], ["TOO_MANY_MEDIA"],
                           f"{len(media)} media files, only the first {CFG.MAX_MEDIA_PER_POST} were checked"))
    if not items:
        items.append(_item("post", "post", "REVIEW", [], ["EMPTY_POST"], "post has no text and no media"))

    decision, cats, reasons, why = merge_items(items)
    return {"post_id": post["id"], "decision": decision, "categories": cats, "reasons": reasons, "explanation": why,
            "items": items, "seconds": round(time.perf_counter() - t0, 2), "model": dict(MODEL_INFO)}

def process_post(user_id, post_id):
    """fetch + moderate. Raises on transient problems (network, post not visible yet) so the job can be retried."""
    post = find_post(user_id, post_id)
    if post is None:
        raise LookupError(f"post {post_id} not found for user {user_id}")
    if post["deleted"]:
        raise PostGone(f"post {post_id} was deleted")
    res = moderate_post(post)
    res["user_id"] = user_id
    return res

## 9b. Token check (run this if the fetch answers 401)
Shows whose token it is, whether it expired, and whether it can read `CFG.USER_ID`. The token itself is never printed.

In [10]:
# new
# ===== Token check: whose token is it, is it expired, and can it read USER_ID? (the token itself is never printed) =====
import base64, time, requests

def decode_jwt(tok):
    """read the claims of a JWT (no secret needed). Returns {} if the token is not a JWT."""
    try:
        part = tok.split(".")[1]
        part += "=" * (-len(part) % 4)
        return json.loads(base64.urlsafe_b64decode(part))
    except Exception:
        return {}

def token_user_id(claims):
    for k in ("userId", "user_id", "uid", "id", "sub"):
        if claims.get(k) not in (None, ""):
            return claims[k]
    return None

def _status(user_id):
    """HTTP status of GET .../user/{user_id}/skills-posts with the current token (None = network error)"""
    url = CFG.API_BASE.rstrip("/") + CFG.POSTS_ENDPOINT.format(user_id=user_id)
    try:
        return requests.get(url, headers=_headers(), timeout=CFG.REQUEST_TIMEOUT).status_code
    except Exception as e:
        print("   network problem:", type(e).__name__, str(e)[:120])
        return None

def diagnose_token(verbose=True):
    print("token set:", bool(API_TOKEN), "| length:", len(API_TOKEN or ""))
    if not API_TOKEN:
        print("-> No token. Put EVOLVE_TOKEN in Colab Secrets (notebook access ON) and re-run the Config cell.")
        return
    claims = decode_jwt(API_TOKEN)
    uid = token_user_id(claims)
    if claims:
        print("claims   :", {k: v for k, v in claims.items() if k in ("sub", "id", "userId", "user_id", "uid", "role", "roles", "email", "username", "iat", "exp")})
        if "exp" in claims:
            left = claims["exp"] - time.time()
            print("expires  :", time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime(claims["exp"])),
                  "-> EXPIRED" if left < 0 else f"-> valid for another {left/3600:.1f} h")
        print("token user:", uid)
    else:
        print("The token is not a readable JWT (opaque token), so its user cannot be shown.")

    s = _status(CFG.USER_ID)
    print(f"GET user {CFG.USER_ID} -> HTTP {s}")
    if s == 200:
        print("-> OK: the backend accepts this token for this user."); return
    if uid is not None and str(uid) != str(CFG.USER_ID):
        s2 = _status(uid)
        print(f"GET user {uid} (the token's own user) -> HTTP {s2}")
        if s2 == 200:
            print(f"-> The token is valid but it only reads ITS OWN data (user {uid}). "
                  f"Set CFG.USER_ID = {uid}, or log in as user {CFG.USER_ID}, or ask for an admin/service token.")
            return
        if s2 in (401, 403, 404):
            print(f"-> Even the token's own user ({uid}) is refused. The token is expired, or user {uid} no longer exists in the backend "
                  "(deleted users cannot log in). Log in again with an account that still exists and paste that NEW token.")
            return
    if s in (401, 403):
        print("-> Token rejected. Likely expired / belongs to a deleted user / the route needs an admin role. Log in again and use the new token.")
    elif s == 404:
        print(f"-> User {CFG.USER_ID} was not found in the backend. Use a USER_ID that exists.")

diagnose_token()


token set: True | length: 183
claims   : {'sub': 3, 'email': 'ankitmor9600@gmail.com', 'iat': 1792584887, 'exp': 1792588487}
expires  : 2026-10-21 13:14:47 UTC -> valid for another 0.7 h
token user: 3
GET user 3 -> HTTP 200
-> OK: the backend accepts this token for this user.


## 10. Fetch the user's posts from the backend

In [11]:
# changed
# ===== Fetch the user's posts from the backend (on 401/403 it now runs the token check instead of just guessing) =====
import pandas as pd
url = CFG.API_BASE.rstrip("/") + CFG.POSTS_ENDPOINT.format(user_id=CFG.USER_ID)
print("GET", url)
try:
    DATA = fetch_user_data(CFG.USER_ID)
except requests.HTTPError as e:
    code = e.response.status_code
    if code in (401, 403, 404):
        print(f"Backend answered HTTP {code}. Running the token check:\n")
        diagnose_token()
        raise RuntimeError(f"Backend answered HTTP {code} for user {CFG.USER_ID}. Read the token check above: "
                           "usually the token is expired, belongs to a deleted user, or cannot read this user's data.") from None
    raise RuntimeError(f"Backend answered HTTP {code}. " + e.response.text[:200]) from None

POSTS = [normalize_post(p, i) for i, p in enumerate(DATA.get("posts", []))]
if CFG.POST_IDS:
    POSTS = [p for p in POSTS if str(p["id"]) in {str(x) for x in CFG.POST_IDS}]
print(f"user {(DATA.get('user') or {}).get('id', CFG.USER_ID)} ({(DATA.get('user') or {}).get('name', '')}) | {len(POSTS)} post(s)")

display(pd.DataFrame([{"post_id": p["id"], "type": p["type"], "caption": p["text"][:70], "media": len(p["media"]),
                       "media_types": ",".join((m["type"] or "?") for m in p["media"]), "deleted": p["deleted"]} for p in POSTS]))


GET https://evolve-backend-4lux.onrender.com/api/skills/user/3/skills-posts
user 3 (ankitmor) | 8 post(s)


,post_id,type,caption,media,media_types,deleted
0,6,IMAGE,Batch Test 2,1,IMAGE,False
1,7,VIDEO,Batch Tes 5,1,VIDEO,False
2,9,TEXT,My document,1,DOCUMENT,False
3,11,VIDEO,Batch Tes 5,1,VIDEO,False
4,13,VIDEO,Batch Tes 5,1,VIDEO,False
5,14,IMAGE,Batch Test 2,1,IMAGE,False
6,18,IMAGE,Batch Test 2,1,IMAGE,False
7,19,IMAGE,Batch Test 2,1,IMAGE,False


In [12]:
# new
# Every post with its media URLs (nothing is cut off)
for p in DATA["posts"]:
    print(f"post {p['id']} | {p['postType']} | caption: {p.get('caption')!r}")
    for m in p.get("media") or []:
        print("   ", m.get("mediaType"), "->", m.get("mediaUrl"))

post 6 | IMAGE | caption: 'Batch Test 2'
    IMAGE -> https://res.cloudinary.com/dh3de6exr/image/upload/v1792415083/evolv/posts/images/post_6/lvlglv3cfcz0kfjjncqm.jpg
post 7 | VIDEO | caption: 'Batch Tes 5'
    VIDEO -> https://res.cloudinary.com/dh3de6exr/video/upload/v1792415126/evolv/posts/videos/post_7/fkhrfm4kfvabvr0eo15l.mp4
post 9 | TEXT | caption: 'My document'
    DOCUMENT -> https://res.cloudinary.com/dh3de6exr/raw/upload/v1792415352/evolv/posts/documents/post_9/1792415352143_Ankit_Mor_Resume__1_.pdf
post 11 | VIDEO | caption: 'Batch Tes 5'
    VIDEO -> https://res.cloudinary.com/dh3de6exr/video/upload/v1792475133/evolv/posts/videos/post_11/fh2n9dmpexjxyg37nfoz.mp4
post 13 | VIDEO | caption: 'Batch Tes 5'
    VIDEO -> https://res.cloudinary.com/dh3de6exr/video/upload/v1792500677/evolv/posts/videos/post_13/qfi9mrbjenoi1sfzn4xg.mp4
post 14 | IMAGE | caption: 'Batch Test 2'
    IMAGE -> https://res.cloudinary.com/dh3de6exr/image/upload/v1792501456/evolv/posts/images/post_14/vhp8

## 11. Moderate every post

In [13]:
# ===== Moderate every fetched post =====
t0 = time.perf_counter()
RESULTS = []
for p in POSTS:
    if p["deleted"]:
        print(f"post {p['id']}: deleted, skipped"); continue
    res = moderate_post(p)
    res["user_id"] = CFG.USER_ID
    RESULTS.append(res)
    print(f"post {p['id']:>4}: {res['decision']:7} {res['seconds']:>6}s  {res['explanation'][:110]}")
print(f"\n{len(RESULTS)} post(s) in {time.perf_counter() - t0:.0f}s")

display(pd.DataFrame([{"post_id": r["post_id"], "decision": r["decision"], "categories": ", ".join(r["categories"]),
                       "reasons": ", ".join(r["reasons"]), "seconds": r["seconds"]} for r in RESULTS]))

out_path = f"/content/moderation_results_user{CFG.USER_ID}.json"
json.dump({"user_id": CFG.USER_ID, "results": RESULTS}, open(out_path, "w"), indent=2, ensure_ascii=False)
print("Saved", out_path)

post    6: SAFE      7.04s  No unsafe content found in 2 item(s).
post    7: SAFE      9.42s  No unsafe content found in 2 item(s).
post    9: REVIEW    1.73s  media 1: REVIEW (Unsupported file type: )
post   11: SAFE      8.16s  No unsafe content found in 2 item(s).
post   13: SAFE      8.14s  No unsafe content found in 2 item(s).
post   14: UNSAFE     7.6s  media 1: UNSAFE (Flagged in: image (Violence))
post   18: UNSAFE    7.89s  media 1: UNSAFE (Flagged in: image (Violence))
post   19: UNSAFE    7.91s  media 1: UNSAFE (Flagged in: image (Violence))

8 post(s) in 58s


,post_id,decision,categories,reasons,seconds
0,6,SAFE,,,7.04
1,7,SAFE,,,9.42
2,9,REVIEW,,UNSUPPORTED_TYPE,1.73
3,11,SAFE,,,8.16
4,13,SAFE,,,8.14
5,14,UNSAFE,Violence,FLAGGED_IMAGE,7.60
6,18,UNSAFE,Violence,FLAGGED_IMAGE,7.89
7,19,UNSAFE,Violence,FLAGGED_IMAGE,7.91


Saved /content/moderation_results_user3.json


### Details per post

In [14]:
# Look inside one post: every item (caption, each media file) with its own decision
def show_post(res):
    print(f"post {res['post_id']}: {res['decision']}  | {res['explanation']}")
    for it in res["items"]:
        print(f"  {it['item']:9} {it['type']:6} {it['decision']:7} {it['seconds']:>6}s  {it['reasons'] or ''} {it['categories'] or ''}")
        for k, v in (it.get("details") or {}).items():
            print(f"      [{k}] {str(v)[:200]}")

for r in RESULTS:
    show_post(r)

post 6: SAFE  | No unsafe content found in 2 item(s).
  caption   text   SAFE      1.72s   
  media 1   image  SAFE      5.26s   
post 7: SAFE  | No unsafe content found in 2 item(s).
  caption   text   SAFE      1.64s   
  media 1   video  SAFE      6.86s   
post 9: REVIEW  | media 1: REVIEW (Unsupported file type: )
  caption   text   SAFE      1.67s   
  media 1   unknown REVIEW     0.0s  ['UNSUPPORTED_TYPE'] 
post 11: SAFE  | No unsafe content found in 2 item(s).
  caption   text   SAFE      1.68s   
  media 1   video  SAFE      6.41s   
post 13: SAFE  | No unsafe content found in 2 item(s).
  caption   text   SAFE       1.7s   
  media 1   video  SAFE      6.37s   
post 14: UNSAFE  | media 1: UNSAFE (Flagged in: image (Violence))
  caption   text   SAFE      1.72s   
  media 1   image  UNSAFE    5.83s  ['FLAGGED_IMAGE'] ['Violence']
post 18: UNSAFE  | media 1: UNSAFE (Flagged in: image (Violence))
  caption   text   SAFE      1.78s   
  media 1   image  UNSAFE    5.94s  ['FLAGGED_

## 12. Host it: an API on Colab the platform can call
Flow: the backend sends `POST {user_id, post_id}` to the webhook URL, this notebook fetches that post, moderates it, and (once enabled) sends the decision back.

Before you run it, add `WEBHOOK_SECRET` (any long random string) in Colab Secrets, notebook access ON. Run order: the API cell, the start cell, then the test cell.

In [15]:
# new
# ===== API server around moderate_post(): the platform calls it, the decision can be sent back =====
# Endpoints (all except /health need the header  X-Webhook-Secret):
#   GET  /health             -> status (no secret needed)
#   POST /webhook/new-post   -> {"user_id": 3, "post_id": 6}   queued, answers instantly (the platform uses this)
#   POST /check              -> {"user_id": 3, "post_id": 6}   waits and returns the decision (for testing)
#   POST /moderate/text      -> {"text": "..."}                checks one piece of text
#   GET  /results            -> every decision made so far
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"], check=True)
import queue, hmac, threading, uvicorn
from fastapi import FastAPI, Header, HTTPException, Request

# Colab Secrets -> WEBHOOK_SECRET (notebook access ON). Never write the secret inside the notebook.
WEBHOOK_SECRET = get_secret("WEBHOOK_SECRET", "Choose a webhook secret (long random string): ")

SEND_RESULTS_BACK = False   # set True only after the backend has an endpoint that accepts the decision
RESULT_ENDPOINT = "/api/moderation/user/{user_id}/posts/{post_id}"   # PROPOSED: agree the real route and JSON shape with the backend team
MAX_ATTEMPTS = 3            # a brand-new post may not be visible in the API for a few seconds: retry before giving up
RETRY_WAIT_S = (5, 20)

JOBS = queue.Queue()        # one GPU = one worker: jobs wait in line here
RESULTS_BY_POST = {}
app = FastAPI()

def _auth(secret):
    if not hmac.compare_digest((secret or "").encode(), WEBHOOK_SECRET.encode()):
        raise HTTPException(401, "bad secret")

def slim(res):
    """what is returned / sent back: the decision and why, without the heavy debug details"""
    keep = ("item", "type", "decision", "categories", "reasons", "explanation")
    return {"user_id": res.get("user_id"), "post_id": res["post_id"], "decision": res["decision"],
            "categories": res["categories"], "reasons": res["reasons"], "explanation": res["explanation"],
            "items": [{k: i[k] for k in keep} for i in res["items"]],
            "model": res.get("model"), "seconds": res["seconds"]}

def run_job(uid, pid):
    """fetch + moderate one post. Retries when the post is not visible yet or the network hiccups."""
    last = None
    for attempt in range(MAX_ATTEMPTS):
        try:
            return process_post(uid, pid)
        except PostGone as e:
            return {"user_id": uid, "post_id": pid, "status": "deleted", "error": str(e)}
        except requests.HTTPError as e:
            if e.response.status_code in (401, 403):      # a retry will not fix a bad token
                return {"user_id": uid, "post_id": pid,
                        "error": f"backend refused the token (HTTP {e.response.status_code}); run diagnose_token()"}
            last = e
        except Exception as e:                            # LookupError (post not there yet), timeouts, ...
            last = e
        if attempt < MAX_ATTEMPTS - 1:
            time.sleep(RETRY_WAIT_S[min(attempt, len(RETRY_WAIT_S) - 1)])
    return {"user_id": uid, "post_id": pid, "error": f"{type(last).__name__}: {last}"}

def send_back(uid, pid, res):
    url = CFG.API_BASE.rstrip("/") + RESULT_ENDPOINT.format(user_id=uid, post_id=pid)
    try:
        requests.post(url, json=slim(res), headers=_headers(), timeout=CFG.REQUEST_TIMEOUT).raise_for_status()
        return True
    except Exception as e:
        print(f"send_back failed for post {pid}:", type(e).__name__, str(e)[:150])
        return False

def worker():
    while True:
        uid, pid = JOBS.get()
        try:
            res = run_job(uid, pid)
            RESULTS_BY_POST[(str(uid), str(pid))] = res
            print(f"[user {uid} | post {pid}] ->", res.get("decision", res.get("error")))
            if SEND_RESULTS_BACK and "decision" in res:
                send_back(uid, pid, res)
        except Exception as e:
            print(f"[user {uid} | post {pid}] worker error:", type(e).__name__, e)
        finally:
            JOBS.task_done()

@app.get("/health")
def health():
    return {"ok": True, "queued": JOBS.qsize(), "done": len(RESULTS_BY_POST), "sending_back": SEND_RESULTS_BACK}

@app.get("/results")
def results(x_webhook_secret: str = Header(default="")):
    _auth(x_webhook_secret)
    return [slim(r) if "decision" in r else r for r in RESULTS_BY_POST.values()]

async def _ids_from(request):
    body = await request.json()
    uid = body.get("user_id")
    pid = body.get("post_id") or (body.get("post") or {}).get("id")
    if uid is None or pid is None:
        raise HTTPException(422, "need user_id and post_id")
    return uid, pid

@app.post("/webhook/new-post")
async def new_post(request: Request, x_webhook_secret: str = Header(default="")):
    _auth(x_webhook_secret)
    uid, pid = await _ids_from(request)
    JOBS.put((uid, pid))
    return {"queued": True, "position": JOBS.qsize()}

@app.post("/check")
def check_now(body: dict, x_webhook_secret: str = Header(default="")):
    _auth(x_webhook_secret)
    uid, pid = body.get("user_id"), body.get("post_id")
    if uid is None or pid is None:
        raise HTTPException(422, "need user_id and post_id")
    res = run_job(uid, pid)
    RESULTS_BY_POST[(str(uid), str(pid))] = res
    if "decision" not in res:
        raise HTTPException(410 if res.get("status") == "deleted" else 502, res.get("error", "failed"))
    return slim(res)

@app.post("/moderate/text")
def moderate_text_api(body: dict, x_webhook_secret: str = Header(default="")):
    _auth(x_webhook_secret)
    text = (body.get("text") or "").strip()
    if not text:
        raise HTTPException(422, "need text")
    with GPU_LOCK:
        r = moderate(text=text)
    return {"decision": r["decision"], "categories": r["categories"], "reasons": r["reasons"], "explanation": r["explanation"]}

print("API defined. Results are sent back to the backend:", SEND_RESULTS_BACK)


Choose a webhook secret (long random string): ··········
API defined. Results are sent back to the backend: False


In [16]:
# new
# ===== Start the server + public URL (run once per Colab session) =====
import subprocess, re
PORT = 8000

if "server" not in globals():                                   # re-running this cell never starts a second server
    server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    threading.Thread(target=worker, daemon=True).start()

if "tunnel" in globals() and tunnel.poll() is None:
    print("Tunnel already running.")
else:
    if not os.path.exists("cloudflared"):                       # free quick tunnel, no account needed
        subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
                       " -O cloudflared && chmod +x cloudflared", shell=True, check=True)
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
                              stderr=subprocess.PIPE, text=True)
    PUBLIC_URL = None
    for line in tunnel.stderr:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            PUBLIC_URL = m.group(0); break
    threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()
    assert PUBLIC_URL, "cloudflared did not give a public URL"

print("Health check :", PUBLIC_URL + "/health")
print("Webhook URL  :", PUBLIC_URL + "/webhook/new-post      (POST {user_id, post_id})")
print("Sync check   :", PUBLIC_URL + "/check                 (POST {user_id, post_id})")
print("Header       : X-Webhook-Secret: <your WEBHOOK_SECRET>")
print("This URL changes every time the tunnel is restarted, and the server stops when the Colab session ends.")


Health check : https://compiled-eur-pension-favorite.trycloudflare.com/health
Webhook URL  : https://compiled-eur-pension-favorite.trycloudflare.com/webhook/new-post      (POST {user_id, post_id})
Sync check   : https://compiled-eur-pension-favorite.trycloudflare.com/check                 (POST {user_id, post_id})
Header       : X-Webhook-Secret: <your WEBHOOK_SECRET>
This URL changes every time the tunnel is restarted, and the server stops when the Colab session ends.


In [17]:
# new
# ===== Test the server from inside Colab before you give the URL to the backend =====
hdr = {"X-Webhook-Secret": WEBHOOK_SECRET}
print("local health :", requests.get(f"http://localhost:{PORT}/health", timeout=15).json())
print("public health:", requests.get(PUBLIC_URL + "/health", timeout=30).json())     # proves the tunnel works

pid = next(p["id"] for p in POSTS if not p["deleted"])          # first post from the fetch cell
r = requests.post(f"http://localhost:{PORT}/check", headers=hdr, json={"user_id": CFG.USER_ID, "post_id": pid}, timeout=900)
print("POST /check  ->", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:2500])

r = requests.post(f"http://localhost:{PORT}/moderate/text", headers=hdr, json={"text": "Have a nice day, everyone!"}, timeout=300)
print("POST /moderate/text ->", r.status_code, r.json())


local health : {'ok': True, 'queued': 0, 'done': 0, 'sending_back': False}
public health: {'ok': True, 'queued': 0, 'done': 0, 'sending_back': False}
POST /check  -> 200
{
  "user_id": 3,
  "post_id": 6,
  "decision": "SAFE",
  "categories": [],
  "reasons": [],
  "explanation": "No unsafe content found in 2 item(s).",
  "items": [
    {
      "item": "caption",
      "type": "text",
      "decision": "SAFE",
      "categories": [],
      "reasons": [],
      "explanation": "No unsafe content found."
    },
    {
      "item": "media 1",
      "type": "image",
      "decision": "SAFE",
      "categories": [],
      "reasons": [],
      "explanation": "No unsafe content found."
    }
  ],
  "model": {
    "safety": "nvidia/Nemotron-3.5-Content-Safety",
    "dtype": "bfloat16",
    "attn": "sdpa",
    "asr": "whisper-large-v3-turbo (faster-whisper)",
    "triage": "openai/clip-vit-base-patch32"
  },
  "seconds": 7.28
}
POST /moderate/text -> 200 {'decision': 'SAFE', 'categories': [], 're

In [25]:
import requests
r = requests.post(
    "https://compiled-eur-pension-favorite.trycloudflare.com/webhook/new-post",
    headers={"X-Webhook-Secret": "evolve-k29x-8fj3-qp71-zt64"},
    json={"user_id": 3, "post_id": 6},
    timeout=30,
)
print(r.status_code, r.json())

200 {'queued': True, 'position': 1}
